# D1 - Civile flussi 2014-2024

Primo notebook di supporto alla discussion su `civile-flussi`.

Taglio scelto:
- trend nazionale del rapporto `definiti / sopravvenuti`
- backlog nazionale a fine anno
- distretti 2024 con rapporto piu basso e piu alto
- trend 2014-2024 per un set corto di distretti rappresentativi

Il notebook usa il MART reale prodotto dal `toolkit`.

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "preprojects").exists() and (candidate / "out").exists():
            return candidate
    raise FileNotFoundError(f"Repo root non trovato partendo da {start}")

REPO_ROOT = find_repo_root(Path.cwd())
DATA_PATH = REPO_ROOT / "out" / "data" / "mart" / "civile_flussi_2014_2024" / "2024" / "mart_summary.parquet"

assert DATA_PATH.exists(), f"MART non trovato: {DATA_PATH}"
DATA_PATH

In [ ]:
con = duckdb.connect()

nazionale = con.execute(
    f"""
    with src as (select * from read_parquet('{DATA_PATH.as_posix()}'))
    select
      anno,
      sopravvenuti_totali,
      definiti_totali,
      round(definiti_totali / nullif(sopravvenuti_totali, 0), 3) as rapporto_ds,
      pendenti_finali_totali,
      pendenti_finali_totali - lag(pendenti_finali_totali) over(order by anno) as delta_pendenti
    from src
    where livello_aggregazione = 'nazionale'
    order by anno
    """
).fetchall()

bottom_2024 = con.execute(
    f"""
    with src as (select * from read_parquet('{DATA_PATH.as_posix()}'))
    select
      distretto,
      sopravvenuti_totali,
      definiti_totali,
      round(definiti_totali / nullif(sopravvenuti_totali, 0), 3) as rapporto_ds,
      pendenti_finali_totali
    from src
    where livello_aggregazione = 'distretto' and anno = 2024
    order by rapporto_ds asc, distretto asc
    limit 5
    """
).fetchall()

top_2024 = con.execute(
    f"""
    with src as (select * from read_parquet('{DATA_PATH.as_posix()}'))
    select
      distretto,
      sopravvenuti_totali,
      definiti_totali,
      round(definiti_totali / nullif(sopravvenuti_totali, 0), 3) as rapporto_ds,
      pendenti_finali_totali
    from src
    where livello_aggregazione = 'distretto' and anno = 2024
    order by rapporto_ds desc, distretto asc
    limit 5
    """
).fetchall()

selected_districts = ['Venezia', 'Trieste', 'Torino', 'Messina', 'Bari']
trend_distretti = con.execute(
    f"""
    with src as (select * from read_parquet('{DATA_PATH.as_posix()}'))
    select
      anno,
      distretto,
      sopravvenuti_totali,
      definiti_totali,
      round(definiti_totali / nullif(sopravvenuti_totali, 0), 3) as rapporto_ds,
      pendenti_finali_totali
    from src
    where livello_aggregazione = 'distretto'
      and distretto in ('Venezia', 'Trieste', 'Torino', 'Messina', 'Bari')
    order by distretto, anno
    """
).fetchall()

nazionale[:3], bottom_2024[:2], top_2024[:2], trend_distretti[:3]

In [ ]:
def fmt_int(value):
    if value is None:
        return "-"
    return f"{int(round(value)):,}".replace(",", ".")

def fmt_float(value, digits=3):
    if value is None:
        return "-"
    return f"{value:.{digits}f}"

def to_markdown(headers, rows, formatters=None):
    formatters = formatters or {}
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        values = []
        for idx, value in enumerate(row):
            formatter = formatters.get(idx)
            values.append(formatter(value) if formatter else str(value))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

headers_naz = ["Anno", "Sopravvenuti", "Definiti", "Rapporto D/S", "Pendenti finali", "Delta pendenti"]
display(Markdown("## Trend nazionale\n" + to_markdown(headers_naz, nazionale, {1: fmt_int, 2: fmt_int, 3: fmt_float, 4: fmt_int, 5: fmt_int})))

headers_dist = ["Distretto", "Sopravvenuti", "Definiti", "Rapporto D/S", "Pendenti finali"]
display(Markdown("## Distretti 2024 con rapporto D/S piu basso\n" + to_markdown(headers_dist, bottom_2024, {1: fmt_int, 2: fmt_int, 3: fmt_float, 4: fmt_int})))
display(Markdown("## Distretti 2024 con rapporto D/S piu alto\n" + to_markdown(headers_dist, top_2024, {1: fmt_int, 2: fmt_int, 3: fmt_float, 4: fmt_int})))

headers_trend = ["Anno", "Distretto", "Sopravvenuti", "Definiti", "Rapporto D/S", "Pendenti finali"]
display(Markdown("## Trend 2014-2024 per distretti selezionati\n" + to_markdown(headers_trend, trend_distretti, {0: fmt_int, 2: fmt_int, 3: fmt_int, 4: fmt_float, 5: fmt_int})))

In [ ]:
anni = [row[0] for row in nazionale]
rapporti = [row[3] for row in nazionale]
pendenti = [row[4] for row in nazionale]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(anni, rapporti, marker="o", color="#0f766e", linewidth=2)
axes[0].axhline(1.0, color="#7f1d1d", linestyle="--", linewidth=1)
axes[0].set_title("Rapporto definiti / sopravvenuti")
axes[0].set_xlabel("Anno")
axes[0].set_ylabel("Rapporto D/S")
axes[0].grid(alpha=0.25)

axes[1].plot(anni, pendenti, marker="o", color="#1d4ed8", linewidth=2)
axes[1].set_title("Pendenti finali nazionali")
axes[1].set_xlabel("Anno")
axes[1].set_ylabel("Procedimenti")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
distretti = [row[0] for row in bottom_2024]
rapporti_bottom = [row[3] for row in bottom_2024]
rapporti_top = [row[3] for row in top_2024]
distretti_top = [row[0] for row in top_2024]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(distretti[::-1], rapporti_bottom[::-1], color="#b91c1c")
axes[0].axvline(1.0, color="#111827", linestyle="--", linewidth=1)
axes[0].set_title("Distretti 2024 con rapporto D/S piu basso")
axes[0].set_xlabel("Rapporto D/S")

axes[1].barh(distretti_top[::-1], rapporti_top[::-1], color="#15803d")
axes[1].axvline(1.0, color="#111827", linestyle="--", linewidth=1)
axes[1].set_title("Distretti 2024 con rapporto D/S piu alto")
axes[1].set_xlabel("Rapporto D/S")

plt.tight_layout()
plt.show()

In [ ]:
trend_by_distretto = {}
for anno, distretto, sopravvenuti, definiti, rapporto_ds, pendenti in trend_distretti:
    trend_by_distretto.setdefault(distretto, {"anni": [], "rapporti": [], "pendenti": []})
    trend_by_distretto[distretto]["anni"].append(anno)
    trend_by_distretto[distretto]["rapporti"].append(rapporto_ds)
    trend_by_distretto[distretto]["pendenti"].append(pendenti)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for distretto, values in trend_by_distretto.items():
    axes[0].plot(values['anni'], values['rapporti'], marker='o', linewidth=2, label=distretto)
axes[0].axhline(1.0, color='#111827', linestyle='--', linewidth=1)
axes[0].set_title('Trend 2014-2024 del rapporto D/S')
axes[0].set_xlabel('Anno')
axes[0].set_ylabel('Rapporto D/S')
axes[0].grid(alpha=0.25)
axes[0].legend(loc='best')

for distretto, values in trend_by_distretto.items():
    axes[1].plot(values['anni'], values['pendenti'], marker='o', linewidth=2, label=distretto)
axes[1].set_title('Trend 2014-2024 dei pendenti finali')
axes[1].set_xlabel('Anno')
axes[1].set_ylabel('Pendenti finali')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Lettura rapida

- Dal 2014 al 2023 il backlog nazionale si riduce in modo quasi continuo.
- Il 2024 e il primo anno della serie con `rapporto D/S < 1`, quindi il saldo nazionale torna leggermente negativo.
- Nel 2024 i distretti non si muovono in modo uniforme: `Venezia`, `Trieste` e `Torino` sono sotto pressione, mentre `Messina` e `Bari` restano sopra `1`.
- Il trend distrettuale aiuta a distinguere segnali strutturali da episodi isolati: `Venezia` e `Trieste` peggiorano soprattutto nel finale di serie, mentre `Messina` e `Bari` restano sopra `1` per quasi tutta la finestra.
- Il rapporto D/S da solo non esaurisce la lettura: va tenuto insieme anche ai pendenti finali e alle discontinuita di classificazione introdotte dal 2021 e dal 2022.